This repo has function calling examples
https://github.com/john-carroll-sw/chat-completions-function-calling-examples/

In [38]:
import os
from dotenv import load_dotenv
import json
# Load environment variables from .env file
load_dotenv()

True

In [39]:
def get_current_weather(location, unit="fahrenheit"):
    """Get the current weather in a given location"""
    if "tokyo" in location.lower():
        return json.dumps({"location": "Tokyo", "temperature": "10", "unit": unit})
    elif "san francisco" in location.lower():
        return json.dumps(
            {"location": "San Francisco", "temperature": "72", "unit": unit}
        )
    elif "paris" in location.lower():
        return json.dumps({"location": "Paris", "temperature": "22", "unit": unit})
    else:
        return json.dumps({"location": location, "temperature": "unknown"})


In [40]:
tools = [
        {
            "type": "function",
            "function": {
                "name": "get_current_weather",
                "description": """
                    Get the current weather in a given location. 
                    Note: any US cities have temperatures in Fahrenheit
                """,
                "parameters": {
                    "type": "object",
                    "properties": {
                        "location": {
                            "type": "string",
                            "description": "The city and state, e.g. San Francisco, CA",
                        },
                        "unit": {
                            "type": "string", 
                            "description": "Unit of Measurement (Celsius or Fahrenheit) for the temperature based on the location",
                            "enum": ["celsius", "fahrenheit"]
                        },
                    },
                    "required": ["location"],
                },
            },
        }
    ]

In [ ]:
def demo_openai():
    """Demonstrate text generation using the OpenAI API."""
    print("="*50)
    print("Initializing OpenAI API...")
    
    # Check if API key is present
    if not os.environ.get("OPENAI_API_KEY"):
        print("Error: OPENAI_API_KEY is not set in the environment.")
        return

    try:
        from openai import OpenAI
        client = OpenAI(base_url="https://generativelanguage.googleapis.com/v1beta/openai/",api_key="")
        
        prompt = "What's the weather like in San Francisco, Tokyo, and Paris?"
        print(f"Prompt: {prompt}\n")
        
        print("Calling OpenAI (gemini)...")
        messages=[
            {"role": "system", "content": """ You are a helpful assistant.
                You have access to a function that can get the current weather in a given location.
                Determine a reasonable Unit of Measurement (Celsius or Fahrenheit) for the temperature based on the location.
                """
            },
            {"role": "user", "content": prompt}
        ]
        response = client.chat.completions.create(
            model="gemini-2.5-flash",
            messages=messages,
            tools = tools,
            max_tokens=500
        )
        
        print("\nOpenAI (Gemini) Response:")
        response_message = response.choices[0].message
        print(response_message.content)
        print(response_message.tool_calls)     
        tool_calls = response_message.tool_calls

        if tool_calls:

            messages.append(response_message)  # extend conversation with assistant's reply
            available_functions = {
                "get_current_weather": get_current_weather,
            }  # only one function in this example, but you can have multiple
        
            for tool_call in tool_calls:

                # Step 3: call the function
                # Note: the JSON response may not always be valid; be sure to handle errors
                function_name = tool_call.function.name

                function_args = json.loads(tool_call.function.arguments)
                # get the function and arguments

                if tool_call.function.name not in available_functions:
                   return "Function " + tool_call.function.name + " does not exist", None

                function_to_call = available_functions[tool_call.function.name]
                # call the function
                function_response = function_to_call(**function_args)

                # Step 4: send the info for each function call and function response to the model
                messages.append(
                    {
                        "tool_call_id": tool_call.id,
                        "role": "tool",
                        "name": function_name,
                        "content": function_response,
                    }
                )  # extend conversation with function response

            second_response = client.chat.completions.create(
                model="gemini-2.5-flash",
                messages=messages,
                temperature=0,  # Adjust the variance by changing the temperature value (default is 0.8)
            )  # get a new response from the model where it can see the function response
            print("Second response")
            print(second_response.choices[0].message.content)        
   

        
    except Exception as e:
        print(f"An error occurred with OpenAI: {e}")

 
        
    print("="*50)

In [42]:
def demo_google_genai():
    """Demonstrate text generation using the Google GenAI SDK."""
    print("="*50)
    print("Initializing Google GenAI API...")
    
    # Check if API key is present
    if not os.environ.get("GEMINI_API_KEY"):
        print("Error: GEMINI_API_KEY is not set in the environment.")
        return

    try:
        from google import genai
        from google.genai import types
        
        # The genai SDK automatically picks up GEMINI_API_KEY from environment variables
        client = genai.Client()
        
        #prompt = "Explain the concept of decorators in Python in one short paragraph."
        prompt = "Who won yesterday match between CSK and MI"
        print(f"Prompt: {prompt}\n")
        
        print("Calling Google GenAI (gemini-2.5-flash)...")
        response = client.models.generate_content(
            model='gemini-2.5-flash',
            contents=prompt,
            config=types.GenerateContentConfig(
               tools=[types.Tool(google_search=types.GoogleSearch())]
            )
        )
        
        print("\nGoogle GenAI Response:")
        print(response.text)
        
    except Exception as e:
        print(f"An error occurred with Google GenAI: {e}")
        
    print("="*50)


In [44]:
if __name__ == "__main__":
    print("Starting LLM APIs Demonstration...")
    
    # Run OpenAI Demo
    demo_openai()
    
    # Run Google GenAI Demo
    demo_google_genai()
    
    print("Demonstration complete.")

Starting LLM APIs Demonstration...
Initializing OpenAI API...
Prompt: What's the weather like in San Francisco, Tokyo, and Paris?

Calling OpenAI (gemini)...
An error occurred with OpenAI: Error code: 403 - [{'error': {'code': 403, 'message': 'Your API key was reported as leaked. Please use another API key.', 'status': 'PERMISSION_DENIED'}}]
Initializing Google GenAI API...
Prompt: Who won yesterday match between CSK and MI

Calling Google GenAI (gemini-2.5-flash)...
An error occurred with Google GenAI: 403 PERMISSION_DENIED. {'error': {'code': 403, 'message': 'Your API key was reported as leaked. Please use another API key.', 'status': 'PERMISSION_DENIED'}}
Demonstration complete.
